# Problem Set: Reviewing rail-ontology change requests with a reasoner and Claude
### Chapter 3 — Description Logics

**Topic coverage:** Keet §3.1 (ALC, the tableau), §3.2 (the DL naming scheme and what each letter costs), §3.3 (reasoning services by reduction to satisfiability); course notebooks 01–04 of this chapter. Tools: DSPy and GEPA (Chapters 1–2), **tool-calling agents** through `agentlab.tools` and `agentlab.agents` (new in this chapter), and the Chapter 3 tableau reasoner.

**Level:** Graduate (practice-first). Combines conceptual analysis with building, measuring and optimising a real Claude-backed agent.

**Total points:** 100 (Part A 20 / Part B 29 / Part C 19 / Part D 20 / Part E 12); 53 pts are auto-graded by the check cells, the rest are marked by hand against the rubric. **Estimated effort:** 10–12 hours.

**API budget:** ≈ \$1–3 **estimated, not yet measured** on `claude-sonnet-5-5` (≈ 160 DSPy calls at ≈ \$0.005 each, GEPA's reflection on `claude-opus-5-5`, and 13 agent episodes at a few cents each; GEPA is the largest line). Your cost lines are the measurement. Re-running unchanged cells is free: DSPy caches identical requests. Every model call in this notebook is a billed request to Anthropic; the cost lines the problems ask for are part of the answer.

## The brief

**Meridian Rail**, a (fictional) railway infrastructure manager, keeps its asset
information in a modular OWL ontology. Discipline teams — track, signalling,
structures, power, stations — submit new modules as pull requests to the
**Ontology Change Board**, which applies two gates before a module is merged:

* **Expressivity.** The nightly classification job runs a lightweight reasoner.
  A module that needs transitive roles, a role hierarchy, inverse roles or number
  restrictions must be routed to the heavyweight reasoner job instead, so every
  request must state the **description logic the module needs** — ALC, S, ALCHIQ, …
* **The reviewer's question.** Each request carries one question from the
  discipline lead — *"is every level-crossing barrier a safety-critical asset?"* —
  which is a **subsumption** query against the module. The board wants a verified
  answer, not an opinion.

The board has asked you, the ontology engineer, to automate the review:

1. **Understand the tools you already have.** A cheap *told-subsumption* heuristic
   and the Chapter 3 tableau. The tableau is a **free, always-right oracle** on
   this corpus — which makes this the one chapter where a deployed system can
   label its own failures with no human in the loop. Find out exactly where
   "always right" stops.
2. **Build the grader and a Claude reviewer**, measure it on a development split.
3. **Optimise it honestly** with GEPA (train / dev / held-out test, cost, noise).
4. **Ship a reviewer that cannot guess**: an agent that calls the reasoner, and a
   pipeline in which the model proposes and the reasoner disposes — with an
   escalation to a human where the reasoner cannot certify its answer.
5. **Check that you measured reasoning, not memory.** Run the reviewer on the Pizza
   ontology every model has read — famous, renamed, and with the axioms changed —
   and on an ontology no model has read: the fleet ontology SIP builds.

The corpus (24 labelled modules, split 8/8/8 by item — one per phenomenon per
split, four "subsumed" and four "not subsumed" in every split — plus 6 unlabelled
field requests), the review guidelines, the agent's tools, the contamination twins
and SIP's fleet modules are provided in `ch03_agentic.py`.

> The original brief also **priced the reasoner** with a budgeted, stochastic MDP and
> let the reviewer **improve itself** on oracle-labelled traffic. They return in
> Chapter 6 (MDPs) and Chapter 10 (skills and gated self-improvement), the chapters
> that introduce those layers.

## Submission requirements

1. **This notebook, executed top to bottom**, with every `TODO` implemented and every *Your answer* cell written.
2. **The artefacts it saves** under `artifacts/` (optimised instructions, run logs) — they are evidence, not by-products.
3. **Honest reporting.** A number without its split, its sample size and its cost earns no credit. Report the runs you did, including the ones that did not improve anything.

The check cells are the marking scheme for the auto-graded points; do not edit them. The final self-test prints your auto-graded subtotal.

> Solutions: `05_solutions.ipynb` — do not open it until you have attempted every problem.

In [ ]:
import sys, json, logging
from pathlib import Path

# Make the course root (the folder holding agentlab/) and this chapter importable.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "agentlab").is_dir():
        sys.path.insert(0, str(candidate))
        break
sys.path.insert(0, str(here))

import agentlab
from agentlab import llm
from agentlab.grading import Grader

print(json.dumps(agentlab.describe_environment(), indent=1))
llm.require_credentials()             # this problem set calls the live Anthropic API
logging.getLogger("dspy").setLevel(logging.WARNING)
GRADER = Grader('Chapter 3 — Description Logics — Reviewing rail-ontology change requests with a reasoner and Claude')

import re, statistics as st
import dspy
import numpy as np
import pandas as pd
import ch03_toolkit as dl
import ch03_agentic as AG
from agentlab import evaluation as ev, optimize as opt, agents

A = dl.Atomic
train, dev, test = AG.build_dataset("train"), AG.build_dataset("dev"), AG.build_dataset("test")
CERTIFIED = AG.validate_gold()          # every label re-derived by the engine (a few seconds)
print(f"train {len(train)} / dev {len(dev)} / test {len(test)} / field {len(AG.FIELD_CASES)}")

### The corpus

Every row is one change request. `kb` is what the reviewer (human or Claude) sees — the
module's axioms and role facts, rendered *without* naming the logic — and `query` is the
discipline lead's question. The gold labels were written by hand and then re-derived by
`AG.validate_gold()`, which also tells you **how** each verdict is certified.

In [ ]:
overview = pd.DataFrame([{"id": ex.id, "split": ex.split, "phenomenon": ex.phenomenon,
                          "gold_dl": ex.gold_dl, "subsumed": ex.gold_subsumption,
                          "certified by": CERTIFIED[ex.id]} for ex in train + dev + test])
print(overview.groupby("split").subsumed.agg(["count", "sum"]).rename(columns={"sum": "n_subsumed"}))
overview

In [ ]:
ex = dev[0]
print(ex.kb, "\n")
print(ex.query)

---

## Part A — The cheap heuristic, the free oracle, and where it stops being right

Before anything touches a model, understand the two answers you can already compute for
free: a structural heuristic, and the tableau.

### A1. A told-subsumption heuristic, and a dataset that can punish it (10 pts)

Reviewers' first instinct is to *read the hierarchy*: follow the asserted superclasses
upwards and see whether the queried class turns up. Implement that as
`told_subsumes(tbox, sub, sup) -> bool`:

* `sub` is told-subsumed by itself;
* for an axiom whose **left side is atomic** `X`, every **atomic conjunct** of the right
  side (flatten nested `dl.And`) is a told superclass of `X`; anything else on the right
  (`exists`, `forall`, `or`, `not`, number restrictions) is ignored;
* an **equivalence** `X == Y` with *both* sides atomic works in both directions;
* take the transitive closure.

Then build `a1`, one row per labelled item (train + dev + test), with columns `id`,
`split`, `phenomenon`, `gold`, `told` (your heuristic's verdict), `heuristic_correct` and
`risk_class` (`AG.risk_class(tbox)`), and `a1_acc`: indexed by split, with columns
`heuristic` (your heuristic's accuracy) and `always_subsumed` (the accuracy of the trivial
strategy *"always answer subsumed"*).

In writing: (i) the heuristic can only err in one direction — which, and why? (ii) The
old version of this corpus had almost only "subsumed" questions. What would *"always
answer subsumed"* have scored there, and what would that have meant for an optimiser whose
feedback includes the guideline `use-the-reasoner`?

In [ ]:
def told_subsumes(tbox: dl.TBox, sub: str, sup: str) -> bool:
    # TODO
    raise NotImplementedError

a1 = None        # TODO: one row per labelled item
a1_acc = None    # TODO: per split: heuristic accuracy, always-subsumed accuracy

In [ ]:
# Marking scheme for A1 -- do not edit.
with GRADER.check('A1', points=5):
    T = dl.TBox()
    T.add(A("X"), dl.And(A("Y"), dl.And(A("Z"), dl.Exists("r", A("W")))))
    T.add(A("Z"), A("P"))
    T.add(A("P"), A("Q"), equivalence=True)
    T.add(A("K"), dl.And(A("X"), A("Y")), equivalence=True)
    assert told_subsumes(T, "X", "X") and told_subsumes(T, "X", "Z") and told_subsumes(T, "X", "Q")
    assert told_subsumes(T, "Q", "P"), "an atomic equivalence works both ways"
    assert not told_subsumes(T, "X", "W") and not told_subsumes(T, "Y", "X")
    assert not told_subsumes(T, "X", "K"), "a complex definition is not a told superclass"
    for c in AG.CASES + AG.FIELD_CASES:          # a told subsumption is always a real one
        t = c.tbox()
        if told_subsumes(t, c.sub, c.sup):
            assert dl.subsumes(A(c.sub), A(c.sup), t), f"{c.id}: heuristic is unsound"
    assert isinstance(a1, pd.DataFrame) and len(a1) == 24
    assert {"id", "split", "phenomenon", "gold", "told", "heuristic_correct", "risk_class"} <= set(a1.columns)
    assert (a1.heuristic_correct == (a1.told == a1.gold)).all()
    assert set(a1_acc.index) == {"train", "dev", "test"}
    assert np.allclose(a1_acc["always_subsumed"], 0.5)
    assert np.allclose(a1_acc["heuristic"], a1.groupby("split").heuristic_correct.mean()[a1_acc.index])

**Your answer:**

*(Write your answer here.)*

### A2. Where the free oracle stops being right (10 pts)

The tableau decides **ALC**. It *names* S, H, I, N and Q but reasons as if transitivity,
sub-roles, inverses and number restrictions were not there — so it reasons over a
*weaker* theory than the module states.

(a) Build `blind_spot = (tbox, sub, sup)`: a small rail module with a **role hierarchy or
a transitive role** in which `sub ⊑ sup` **is** entailed but `dl.subsumes` says it is
not. Then build `compiled`: a copy of the module plus one or more ALC axioms that the role
box licenses (see `AG.rbox_licensed` for the three accepted schemata), after which the
tableau finds the subsumption.

(b) `AG.validate_gold()` certifies "not subsumed" on the non-ALC modules with
countermodels (`AG.CERTIFICATES`). Write your own certificate `my_certificate` — an
`AG.Interpretation` — for a **pure-ALC** "not subsumed" item of your choice
(`cert_case`, one not in `AG.CERTIFICATES`), and confirm it with `AG.is_countermodel`.

(c) In writing: in which direction is the tableau's verdict trustworthy on *any* module
of this corpus, and why? What does (a) mean for a system that labels its own training
data with this oracle? What would you run in production instead?

In [ ]:
blind_spot = None       # TODO (a): (tbox, sub, sup)
compiled = None         # TODO (a): the module plus RBox-licensed ALC axioms

cert_case = "..."       # TODO (b)
my_certificate = None   # TODO (b): AG.Interpretation(domain=..., concepts=..., roles=...)

raise NotImplementedError

In [ ]:
# Marking scheme for A2 -- do not edit.
with GRADER.check('A2', points=4):
    t, sub, sup = blind_spot
    assert t.role_hierarchy or t.transitive_roles, "the blind spot needs an RBox feature"
    assert not dl.subsumes(A(sub), A(sup), t), "the tableau must miss it"
    assert dl.subsumes(A(sub), A(sup), compiled), "the compiled module must expose it"
    extra = [ax for ax in compiled.axioms if ax not in t.axioms]
    assert extra and all(ax in compiled.axioms for ax in t.axioms)
    assert all(AG.rbox_licensed(ax, t) for ax in extra), "only RBox-licensed axioms may be added"
    c = AG.case(cert_case)
    assert c.gold_subsumed is False and cert_case not in AG.CERTIFICATES
    assert AG.is_countermodel(c.tbox(), c.sub, c.sup, my_certificate)

**Your answer:**

*(Write your answer here.)*

---

## Part B — Build the grader and the Claude reviewer

The guidelines the scorer can report — ids plus the sentence GEPA's reflection reads —
are in `AG.DL_RULEBOOK`:

In [ ]:
for rule in AG.DL_RULEBOOK:
    print(f"{rule.id:38s} {rule.description[:80]}")

### B1. A diagnostic review scorer (14 pts)

Implement `dl_scorer(gold, pred) -> ev.ScoreReport`. `gold` is a dataset row
(`gold_dl`, `gold_subsumption`, `constructors`, `sub_satisfiable`, `sub`, `sup`); `pred`
has string fields `dl` and `subsumption`. The score is **0.5 for the name + 0.5 for the
verdict**.

**Name.** Normalise (strip, upper-case, remove spaces, drop a trailing `(D)`) and parse
against the standard order `^(ALC|S)(H?)(O?)(I?)([QNF]?)$`. Full marks only when the
parsed letters equal gold's. When wrong, report every rule that applies:

| situation | violated |
|---|---|
| does not parse (prose, wrong letter order, …) | `name-the-logic-exactly` |
| base differs (ALC vs S) | `s-for-transitive` |
| H / I needed but missing | `h-for-role-hierarchy` / `i-for-inverse` |
| gold has Q or N and the answer's number letter differs | `q-for-qualified-number` |
| a letter (H, O, I, or a number letter) that gold does not have | `no-unused-letters` |
| wrong, but none of the above applies | `name-the-logic-exactly` |

**Verdict.** Accept `true/yes/subsumed/entailed` and `false/no/not subsumed/not entailed`
(case-insensitive, surrounding punctuation ignored). Anything else earns 0 and
`answer-true-or-false`. A wrong verdict earns 0 and `use-the-reasoner`, plus
`unsatisfiable-subsumed-by-everything` when gold is *subsumed* because the subject class
is unsatisfiable.

The notes must say what was wrong **and why** — for a wrong name, the correct name and
the constructors the module uses. That text is what GEPA reads.

In [ ]:
DL_PATTERN = re.compile(r"^(ALC|S)(H?)(O?)(I?)([QNF]?)$")

def parse_dl(text) -> dict | None:
    # TODO: normalise, then match DL_PATTERN; return the letters, or None
    raise NotImplementedError

def dl_scorer(gold, pred) -> ev.ScoreReport:
    # TODO
    raise NotImplementedError

In [ ]:
# Marking scheme for B1 -- do not edit.
with GRADER.check('B1', points=11):
    by_id = {ex.id: ex for ex in train + dev + test}
    P = lambda d, s: dspy.Prediction(dl=d, subsumption=s)
    cases = [
        ("interlocking-control-centre", "SHIQ", "false", 1.0, []),
        ("interlocking-control-centre", " shiq(D) ", "False.", 1.0, []),
        ("interlocking-control-centre", "ALCHIQ", "no", 0.5, ["s-for-transitive"]),
        ("interlocking-control-centre", "SHIN", "false", 0.5, ["q-for-qualified-number"]),
        ("interlocking-control-centre", "SIQ", "false", 0.5, ["h-for-role-hierarchy"]),
        ("interlocking-control-centre", "SHOIQ", "false", 0.5, ["no-unused-letters"]),
        ("interlocking-control-centre", "The logic is SHIQ", "false", 0.5, ["name-the-logic-exactly"]),
        ("interlocking-control-centre", "SHIQ", "true", 0.5, ["use-the-reasoner"]),
        ("interlocking-control-centre", "SHQ", "probably", 0.0, ["i-for-inverse", "answer-true-or-false"]),
        ("bay-platform-low-use", "ALCQ", "false", 0.5, ["q-for-qualified-number"]),
        ("bay-platform-low-use", "ALC", "false", 0.5, ["q-for-qualified-number"]),
        ("barrier-safety-critical", "ALCN", "true", 0.5, ["no-unused-letters"]),
        ("barrier-safety-critical", "S", "yes", 0.5, ["s-for-transitive"]),
        ("automated-manual-crossing", "ALC", "false", 0.5,
         ["use-the-reasoner", "unsatisfiable-subsumed-by-everything"]),
        ("drainage-culvert", "ALC", "true", 0.5, ["use-the-reasoner"]),
    ]
    for item, d, s, score, violated in cases:
        r = dl_scorer(by_id[item], P(d, s))
        assert abs(r.score - score) < 1e-9 and r.violated == violated, (item, d, s, r.score, r.violated)
        assert all(v in AG.DL_RULEBOOK for v in r.violated)
    wrong = dl_scorer(by_id["interlocking-control-centre"], P("ALCHIQ", "false"))
    assert any("SHIQ" in n and "transitive" in n for n in wrong.notes), "say the right name and why"

### B2. The reviewer as a DSPy program (7 pts)

Write a signature `ModuleReview` with inputs `kb` and `query` and outputs
`justification`, `dl` and `subsumption` (all strings), and a factory
`DLReviewer(instruction)` returning a `dspy.Module` with a single `dspy.Predict` whose
instruction is `instruction`. The field descriptions are part of the prompt — write them
for a colleague who has never seen this notation (manual marks), and say in a sentence why
you put the output fields in the order you did. Configure Claude and run it once on
`train[0]`.

In [ ]:
BASELINE_INSTRUCTION = ("You review ontology modules for a railway. Say which description "
                        "logic the module needs and answer the subsumption question.")

# TODO: class ModuleReview(dspy.Signature): ...
# TODO: def DLReviewer(instruction=BASELINE_INSTRUCTION): ...

lm = llm.configure_dspy()
smoke = None     # TODO: DLReviewer()(**train[0].inputs())

raise NotImplementedError

In [ ]:
# Marking scheme for B2 -- do not edit.
with GRADER.check('B2', points=4):
    assert set(ModuleReview.input_fields) == {"kb", "query"}
    assert {"dl", "subsumption", "justification"} <= set(ModuleReview.output_fields)
    program = DLReviewer("custom instruction")
    assert len(list(program.named_predictors())) == 1
    assert opt.instruction_of(program) == "custom instruction"
    assert isinstance(smoke.dl, str) and isinstance(smoke.subsumption, str)

**Your answer:**

*(Write your answer here.)*

### B3. Baseline on the development split, with its cost (8 pts)

Evaluate the baseline reviewer on `dev` with `ev.evaluate_dataset`, inside
`llm.meter(lm)`. Store the result in `baseline_dev` and the cost in `baseline_dev_cost`,
and build `b3`: the per-item rows joined with each item's `phenomenon` and `gold_dl`.

Then write a short **error analysis** separating the two halves of the task: which
*naming* rules were broken and on which phenomena, and which *verdicts* were wrong — and
for those, was it a definition, an inverse, or the unsatisfiable subject? Which of the two
halves could a tool fix completely, and which needs the model to reason?

In [ ]:
# TODO: baseline_dev = ..., baseline_dev_cost = ..., b3 = ...

raise NotImplementedError

In [ ]:
# Marking scheme for B3 -- do not edit.
with GRADER.check('B3', points=3):
    assert baseline_dev["n"] == len(dev) == 8
    assert {r["item"] for r in baseline_dev["rows"]} == {ex.id for ex in dev}
    assert 0.0 <= baseline_dev["mean_score"] <= 1.0
    assert {"calls", "usd"} <= set(baseline_dev_cost)
    assert isinstance(b3, pd.DataFrame) and len(b3) == 8 and {"phenomenon", "score"} <= set(b3.columns)

**Your answer:**

*(Write your answer here.)*

---

## Part C — Optimise honestly — then let the oracle label the training data

GEPA rewrites the instruction by reflecting on the scorer's feedback. The rules: **optimise
on `train`, select on `dev`, report on `test`**, and put a cost and a noise estimate next
to every number. Then use the chapter's distinctive asset — an oracle that labels for free
— to improve the reviewer from unlabelled traffic, behind a promotion gate.

### C1. A budgeted GEPA run with a held-out report (12 pts)

1. Build the GEPA feedback metric from your scorer and `AG.DL_RULEBOOK`, and a separate
   reflection LM (`llm.reflection_lm()`).
2. Run `opt.run_gepa` on `train` with `valset=dev` and `max_metric_calls=GEPA_BUDGET`
   inside `llm.meter(lm, reflect)`; store the cost in `gepa_cost`.
3. Compare baseline and tuned on **`test`** with `opt.compare` (store as `c1`) and print
   `c1.report()`.
4. Save the tuned instruction to `agentlab.artifacts_dir() /
   "ch03_module_reviewer_instruction.txt"`.

In writing: read the instruction diff. Which guidelines did GEPA put into words? Did it
learn to *follow definitions* and to treat an unsatisfiable subject as subsumed — or only
the letter rules? Did it add anything that only fits the eight training modules?

In [ ]:
GEPA_BUDGET = 60
# TODO: gepa_metric, reflect, tuned (inside llm.meter -> gepa_cost), c1, save the instruction
instruction_path = agentlab.artifacts_dir() / "ch03_module_reviewer_instruction.txt"

raise NotImplementedError

In [ ]:
# Marking scheme for C1 -- do not edit.
with GRADER.check('C1', points=6):
    ids = lambda rows: {ex.id for ex in rows}
    assert not (ids(test) & (ids(train) | ids(dev))), "the test split must be untouched"
    assert {r["item"] for r in c1.after["rows"]} == ids(test), "report on the test split"
    assert c1.before["n"] == c1.after["n"] == len(test)
    assert instruction_path.is_file()
    assert instruction_path.read_text(encoding="utf-8") == opt.instruction_of(tuned)
    assert {"calls", "usd"} <= set(gepa_cost) and GEPA_BUDGET <= 100

**Your answer:**

*(Write your answer here.)*

### C2. Is the difference bigger than the noise? (7 pts)

With caching **disabled** (`fresh = llm.dspy_lm(cache=False)`, `with dspy.context(lm=fresh):`),
run the baseline and the tuned reviewers on `test` three times each, inside
`llm.meter(fresh)`; store mean scores in `runs = {"baseline": [...], "gepa": [...]}`.
Set `verdict_c2` to `"significant"` if `|mean_gepa − mean_baseline| > 2·max(sd_gepa,
sd_baseline)`, else `"not significant"`.

In writing: what does this say about the C1 number, and what would a credible evaluation
of the reviewer need?

In [ ]:
fresh = llm.dspy_lm(cache=False)
runs = {"baseline": [], "gepa": []}
# TODO: fill runs (inside llm.meter(fresh) -> c2_cost), then compute verdict_c2
verdict_c2 = None

raise NotImplementedError

In [ ]:
# Marking scheme for C2 -- do not edit.
with GRADER.check('C2', points=4):
    assert set(runs) == {"baseline", "gepa"} and all(len(v) == 3 for v in runs.values())
    assert all(0.0 <= x <= 1.0 for v in runs.values() for x in v)
    _m = {k: st.mean(v) for k, v in runs.items()}
    _s = {k: st.stdev(v) for k, v in runs.items()}
    _expected = ("significant" if abs(_m["gepa"] - _m["baseline"]) > 2 * max(_s.values())
                 else "not significant")
    assert verdict_c2 == _expected
    assert {"calls", "usd"} <= set(c2_cost)

**Your answer:**

*(Write your answer here.)*

---

## Part D — A reviewer that cannot guess

The agent gets the reasoner as tools (`agentlab.tools` logs every call). `check_subsumption`
is the *expensive, sound* call — in production each one is a reasoner job on the integrated
ontology, billed in CI minutes.

In [ ]:
ws_demo = AG.ReviewWorkspace("viaduct-critical")
demo_tools = {t.name: t for t in AG.build_review_tools(ws_demo)}
for name, t in demo_tools.items():
    print(f"{name:22s} {t.description.splitlines()[0]}")
print()
print(demo_tools["dl_expressivity"].invoke({}))
print(demo_tools["check_subsumption"].invoke({"sub": "Viaduct", "sup": "CriticalStructure"}))
print("trajectory:", ws_demo.log.names())

### D1. Build and evaluate the review agent (10 pts)

1. Write `REVIEW_PROMPT`, the agent's system prompt. It must make the agent inspect the
   module, name the logic from the constructors, check whether the subject class is
   satisfiable, **call `check_subsumption` before answering**, and end with a JSON object
   `{"dl": "...", "subsumed": true | false, "evidence": [...]}`.
2. Write `parse_review(text) -> dict` returning `{"dl": str, "subsumption": "true" |
   "false" | "unparseable"}` from the agent's final text (the JSON may be fenced or
   surrounded by prose; take the last object that has a `subsumed` key).
3. For every **test** item: a fresh `AG.ReviewWorkspace(ex.id)`, its tools
   (`AG.build_review_tools(ws)`), an agent (`agents.build_agent(tools,
   system_prompt=...)`), one run (`agents.run_agent(agent, ex.query, log=ws.log)`).
   Collect `d1` with columns `id`, `gold_dl`, `gold`, `predicted_dl`, `predicted`,
   `score` (your `dl_scorer` on the parsed answer), `reasoner_calls` (number of
   `check_subsumption` calls), `tool_calls` and `usd_estimate`
   (`agents.chat_usage(run.messages)`).

In writing: compare the agent with the tuned DSPy reviewer on the same test items — score
**and** cost per request. Which half of the task did the tools fix? Where did the agent
still go wrong, and was it the agent or the reasoner?

In [ ]:
REVIEW_PROMPT = """TODO"""

def parse_review(text: str) -> dict:
    # TODO
    raise NotImplementedError

d1 = None    # TODO: run the agent on every test item

In [ ]:
# Marking scheme for D1 -- do not edit.
with GRADER.check('D1', points=5):
    assert parse_review('{"dl": "SHIQ", "subsumed": false, "evidence": []}') == {"dl": "SHIQ", "subsumption": "false"}
    assert parse_review('Done.\n```json\n{"dl": "ALC", "subsumed": true, "evidence": ["x"]}\n```')["subsumption"] == "true"
    assert parse_review('{"note": 1} and then {"dl": "S", "subsumed": "False", "evidence": []}') == {"dl": "S", "subsumption": "false"}
    assert parse_review("It is subsumed.")["subsumption"] == "unparseable"
    assert isinstance(d1, pd.DataFrame) and list(d1["id"]) == [ex.id for ex in test]
    assert {"id", "gold_dl", "gold", "predicted_dl", "predicted", "score", "reasoner_calls",
            "tool_calls", "usd_estimate"} <= set(d1.columns)
    assert set(d1["predicted"]) <= {"true", "false", "unparseable"}
    by_id = {ex.id: ex for ex in test}
    for _, row in d1.iterrows():
        expected = dl_scorer(by_id[row["id"]],
                             dspy.Prediction(dl=row["predicted_dl"], subsumption=row["predicted"])).score
        assert abs(row["score"] - expected) < 1e-9, row["id"]

**Your answer:**

*(Write your answer here.)*

### D2. The model proposes, the reasoner disposes (10 pts)

The agent of D1 *chooses* to call the reasoner, and a row with `reasoner_calls == 0`
shows it can choose not to. A **pipeline** makes the check unconditional: the reviewer
proposes a name, a verdict and a justification; a deterministic **critic** — the engine —
disposes. This is the pattern the literature on LLMs in ontology engineering converges
on (an LLM proposes, a symbolic tool checks, a human approves), so build it and measure
what each part contributes.

1. Implement `critic(tbox, sub, sup, case_id=None) -> dict` returning `dl` (the engine's
   name), `subsumed` (the tableau's verdict) and `certified`: true when the verdict is
   *subsumed*, or the module is pure ALC (`AG.ALC_CONSTRUCTORS`), or `case_id` has a
   countermodel in `AG.CERTIFICATES` that checks out (A2 told you why).
2. Implement `review_with_critic(program, ex, tbox=None) -> dict`. Run `program` on the
   example; take the module from `tbox` or `AG.case(ex.id).tbox()`; call the critic.
   Return `dl` (the critic's), `subsumption` (`"true"`/`"false"` from the critic when it
   is certified, else `"escalate"`), `source` (`"model+reasoner"` when the model's verdict
   agreed with a certified critic, `"reasoner"` when the critic overrode it,
   `"escalated"` when it could not certify), `disagreed` (did the model's verdict differ
   from the reasoner's?), and the model's raw `model_dl`, `model_subsumption` and
   `justification`.
3. Run it with the **tuned** reviewer from C1 on every **test** item, inside
   `llm.meter(lm)` (store `d2_cost`); collect `d2` with columns `id`, `gold`,
   `model_subsumption`, `subsumption`, `source`, `disagreed` and `score` (your `dl_scorer`
   on `dl` and `subsumption`).
4. Run it on A2's blind spot as `blind_review` (an example built from
   `AG.describe_kb(blind_spot[0])`, with `tbox=blind_spot[0]`).

In writing: put the single-pass reviewer (C1, tuned, test), the agent (D1) and the
pipeline side by side — score, model calls and dollars per request. `disagreed` counts
the model's verdict errors *without any labels*: what is that worth in production? With
a sound and complete checker, what is the model still for — and what happens on a module
where the checker is not complete?

In [ ]:
def critic(tbox: dl.TBox, sub: str, sup: str, case_id: str | None = None) -> dict:
    # TODO
    raise NotImplementedError

def review_with_critic(program, ex, tbox: dl.TBox | None = None) -> dict:
    # TODO
    raise NotImplementedError

d2 = None             # TODO: one row per test item (inside llm.meter(lm) -> d2_cost)
blind_review = None   # TODO: the pipeline on A2's blind spot

In [ ]:
# Marking scheme for D2 -- do not edit.
with GRADER.check('D2', points=5):
    for ex in test:
        c = critic(AG.case(ex.id).tbox(), ex.sub, ex.sup, ex.id)
        assert c["certified"] and c["dl"] == ex.gold_dl and c["subsumed"] == ex.gold_subsumption, ex.id
    _t, _s, _p = blind_spot
    assert critic(_t, _s, _p)["certified"] is False, "an uncertified 'no' must not be trusted"
    assert blind_review["subsumption"] == "escalate" and blind_review["source"] == "escalated"
    assert isinstance(d2, pd.DataFrame) and list(d2["id"]) == [ex.id for ex in test]
    assert {"id", "gold", "model_subsumption", "subsumption", "source", "disagreed",
            "score"} <= set(d2.columns)
    assert set(d2["source"]) <= {"model+reasoner", "reasoner", "escalated"}
    by_id = {ex.id: ex for ex in test}
    for _, row in d2.iterrows():
        ex = by_id[row["id"]]
        assert row["subsumption"] == str(ex.gold_subsumption).lower(), row["id"]
        assert (row["source"] == "reasoner") == bool(row["disagreed"]), row["id"]
    assert {"calls", "usd"} <= set(d2_cost)

**Your answer:**

*(Write your answer here.)*

---

## Part E — Reasoning or remembering?

Every number so far was measured on Meridian Rail's modules, which no model has seen.
Most published results on LLMs in ontology engineering are measured on ontologies every
model *has* seen — the Pizza tutorial above all. Part E measures what that does to a
score, then tests the reviewer on an ontology that was never published: the fleet
ontology SIP builds in its demo.

In [ ]:
pizza = AG.contamination_dataset()
verdicts = pd.DataFrame([{"item": ex.item, "variant": ex.variant,
                          "subsumed": ex.gold_subsumption} for ex in pizza])
print(verdicts.pivot(index="item", columns="variant", values="subsumed")[list(AG.VARIANTS)])
print()
print(pizza[2].kb)
print(pizza[2].query)

### E1. Reasoning or remembering? The Pizza twins (8 pts)

Each of the five questions comes in three variants. **famous**: the tutorial's names and
axioms (miniaturised). **renamed**: the same axioms with names that carry no world
knowledge. **counterfactual**: the tutorial's names with *one* axiom changed, so that the
answer a model remembers is wrong (Margherita without its closure axiom is not provably
vegetarian; an anchovy reclassified as a vegetable makes the Napoletana vegetarian). All
fifteen modules are pure ALC, so every gold verdict is certified.

1. Run the **tuned** reviewer on all fifteen, inside `llm.meter(lm)` (store `e1_cost`), and
   build `e1` with columns `id`, `item`, `variant`, `gold`, `predicted` (`parse_verdict`
   on the answer, or `None`) and `verdict_ok`.
2. `e1_acc`: verdict accuracy per variant (a Series indexed by variant), and
   `contamination_gap = e1_acc["renamed"] - e1_acc["counterfactual"]`.
3. Run the D1 agent on the five **counterfactual** modules
   (`AG.ReviewWorkspace(ex.id, tbox=...)` with the module from `AG.contamination_tbox`),
   and build `e1_agent` with columns `id`, `gold`, `predicted`, `reasoner_calls`.

In writing: what does the gap measure, and what would a 92 % score on the Pizza ontology
(a published figure for LLM-based restriction verification) tell you about a new
ontology? Did the agent answer from its tools or from its memory on the counterfactuals?
What should a benchmark for ontology-engineering assistants contain?

In [ ]:
e1 = None                 # TODO (1), inside llm.meter(lm) -> e1_cost
e1_acc = None             # TODO (2)
contamination_gap = None  # TODO (2)
e1_agent = None           # TODO (3)

raise NotImplementedError

In [ ]:
# Marking scheme for E1 -- do not edit.
with GRADER.check('E1', points=4):
    assert isinstance(e1, pd.DataFrame) and len(e1) == 15
    assert {"id", "item", "variant", "gold", "predicted", "verdict_ok"} <= set(e1.columns)
    assert set(e1["variant"]) == set(AG.VARIANTS)
    gold = {ex.id: ex.gold_subsumption for ex in pizza}
    assert all(row["gold"] == gold[row["id"]] for _, row in e1.iterrows())
    assert (e1.verdict_ok == (e1.predicted == e1.gold)).all()
    assert set(e1_acc.index) == set(AG.VARIANTS)
    assert np.allclose(e1_acc[list(AG.VARIANTS)],
                       e1.groupby("variant").verdict_ok.mean()[list(AG.VARIANTS)])
    assert abs(contamination_gap - (e1_acc["renamed"] - e1_acc["counterfactual"])) < 1e-9
    assert isinstance(e1_agent, pd.DataFrame) and len(e1_agent) == 5
    assert {"id", "gold", "predicted", "reasoner_calls"} <= set(e1_agent.columns)
    assert {"calls", "usd"} <= set(e1_cost)

**Your answer:**

*(Write your answer here.)*

### E2. An ontology no model has read: SIP's fleet (4 pts)

`AG.fleet_dataset()` holds seven review questions over the fleet ontology SIP builds
(vehicles, engines, wheels, manufacturers), with gold certified by the tableau. Evaluate
the **baseline** and the **tuned** reviewers on it with `ev.evaluate_dataset`, and run the
D2 pipeline with the tuned reviewer on every item, all inside `llm.meter(lm)` (store
`e2_cost`). Store `e2 = {"baseline": <mean score>, "tuned": <mean score>,
"pipeline_disagreements": <int>}`.

In writing: did the tuning transfer from rail to fleet? SIP's validate stage now has a
*semantic critic* that asks whether each axiom is **true in the domain**. How is that
question different from the one this reviewer answers, and which of the two can a
reasoner settle?

In [ ]:
fleet = AG.fleet_dataset()
e2 = None     # TODO (inside llm.meter(lm) -> e2_cost)

raise NotImplementedError

In [ ]:
# Marking scheme for E2 -- do not edit.
with GRADER.check('E2', points=2):
    assert len(fleet) == 7 and {ex.split for ex in fleet} == {"fleet"}
    assert set(e2) == {"baseline", "tuned", "pipeline_disagreements"}
    assert 0.0 <= e2["baseline"] <= 1.0 and 0.0 <= e2["tuned"] <= 1.0
    assert isinstance(e2["pipeline_disagreements"], (int, np.integer))
    assert 0 <= e2["pipeline_disagreements"] <= len(fleet)
    assert {"calls", "usd"} <= set(e2_cost)

**Your answer:**

*(Write your answer here.)*

---

## Grading rubric

| Part | Problem | Auto | Manual | Pts |
|---|---|---|---|---|
| A | A1 A told-subsumption heuristic, and a dataset that can punish it | 5 | 5 | 10 |
| A | A2 Where the free oracle stops being right | 4 | 6 | 10 |
| B | B1 A diagnostic review scorer | 11 | 3 | 14 |
| B | B2 The reviewer as a DSPy program | 4 | 3 | 7 |
| B | B3 Baseline on the development split, with its cost | 3 | 5 | 8 |
| C | C1 A budgeted GEPA run with a held-out report | 6 | 6 | 12 |
| C | C2 Is the difference bigger than the noise? | 4 | 3 | 7 |
| D | D1 Build and evaluate the review agent | 5 | 5 | 10 |
| D | D2 The model proposes, the reasoner disposes | 5 | 5 | 10 |
| E | E1 Reasoning or remembering? The Pizza twins | 4 | 4 | 8 |
| E | E2 An ontology no model has read: SIP's fleet | 2 | 2 | 4 |
| **Total** | | | | **100** |

*Manual* points are for the written answers and for the quality of the engineering evidence (splits, sample sizes, cost, honest negative results).

---

## Self-test

Re-run this cell after finishing. It lists every auto-graded problem and whether its check passed.

In [ ]:
AUTO_GRADED = ['A1', 'A2', 'B1', 'B2', 'B3', 'C1', 'C2', 'D1', 'D2', 'E1', 'E2']
_ = GRADER.summary(AUTO_GRADED)